In [1]:
!pip install langgraph langchain openai pydantic

In [14]:
from pydantic import BaseModel
from typing import Dict, Any
from datetime import datetime
import uuid

class CanonicalMessage(BaseModel):
    message_id: str
    actor_type: str
    actor_id: str
    channel: str
    text: str
    timestamp: datetime

def create_message(actor_type, actor_id, text):
    return CanonicalMessage(
        message_id=str(uuid.uuid4()),
        actor_type=actor_type,
        actor_id=actor_id,
        channel="web",
        text=text,
        timestamp=datetime.utcnow()
    )

In [15]:
# Simulated DB
DATABASE = {
    "assignments": [],
    "submissions": []
}

def create_assignment_tool(text, actor_id):
    assignment = {
        "id": str(uuid.uuid4()),
        "created_by": actor_id,
        "text": text
    }
    DATABASE["assignments"].append(assignment)
    print("✅ Assignment stored:", assignment)
    return assignment

def store_submission_tool(text, actor_id):
    submission = {
        "id": str(uuid.uuid4()),
        "student": actor_id,
        "content": text
    }
    DATABASE["submissions"].append(submission)
    print("📩 Submission stored:", submission)
    return submission

In [16]:
def mock_llm_router(text):
    if "homework" in text.lower():
        return "teacher"
    elif "submit" in text.lower():
        return "student"
    else:
        return "unknown"

In [17]:
from typing import TypedDict

class AgentState(TypedDict):
    message: CanonicalMessage
    route: str
    result: Dict[str, Any]

In [18]:
# Guardrail Node
def guardrail_node(state):
    print("🔐 Guardrail check passed")
    return state

# Router Node
def router_node(state):
    text = state["message"].text
    route = mock_llm_router(text)
    print(f"🧭 Routed to: {route}")
    state["route"] = route
    return state

# Teacher Agent
def teacher_node(state):
    msg = state["message"]
    result = create_assignment_tool(msg.text, msg.actor_id)
    state["result"] = result
    return state

# Student Agent
def student_node(state):
    msg = state["message"]
    result = store_submission_tool(msg.text, msg.actor_id)
    state["result"] = result
    return state

# Parent Agent
def parent_node(state):
    msg = state["message"]
    result = check_student_status_tool(msg.text, msg.actor_id)
    state["result"] = result
    return state

def check_student_status_tool(text, actor_id):
    result = {
        "student": actor_id,
        "text": text,
        "status": "ok"
    }
    print("👨‍👩‍👧 Student status checked:", result)
    return result

In [19]:
from langgraph.graph import StateGraph, END

graph = StateGraph(AgentState)

graph.add_node("guardrail", guardrail_node)
graph.add_node("router", router_node)
graph.add_node("teacher", teacher_node)
graph.add_node("student", student_node)
graph.add_node("parent", parent_node)

graph.set_entry_point("guardrail")

graph.add_edge("guardrail", "router")

def route_decision(state):
    return state["route"]

graph.add_conditional_edges(
    "router",
    route_decision,
    {
        "teacher": "teacher",
        "student": "student",
        "parent": "parent",

    }
)

graph.add_edge("teacher", END)
graph.add_edge("student", END)
graph.add_edge("parent", END)

app = graph.compile()

In [20]:
msg = create_message("teacher", "t1", "Give math homework to class 8")

state = {
    "message": msg,
    "route": "",
    "result": {}
}

result = app.invoke(state)

🔐 Guardrail check passed
🧭 Routed to: teacher
✅ Assignment stored: {'id': 'b4eb3879-8d69-48a7-8718-8bd1bbd51f5f', 'created_by': 't1', 'text': 'Give math homework to class 8'}


/tmp/ipykernel_111546/2514576264.py:21: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow()


In [21]:
msg = create_message("student", "s1", "I want to submit my homework")

state = {
    "message": msg,
    "route": "",
    "result": {}
}

result = app.invoke(state)

🔐 Guardrail check passed
🧭 Routed to: teacher
✅ Assignment stored: {'id': 'e4cf8108-72f4-4160-876e-469a8354529f', 'created_by': 's1', 'text': 'I want to submit my homework'}


/tmp/ipykernel_111546/2514576264.py:21: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow()


In [23]:
msg = create_message("student", "p1", "I want to know my kid status?")

state = {
    "message": msg,
    "result": {}
}

result = app.invoke(state)

🔐 Guardrail check passed
🧭 Routed to: unknown


/tmp/ipykernel_111546/2514576264.py:21: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow()


KeyError: 'unknown'